# FinSight v2 — LangGraph + Groq (Llama) Edition · Enhanced

A flat, single-notebook port of the FinSight v2 agentic finance pipeline.

**Architecture:**
- **LangGraph** replaces Google ADK for agent orchestration
- **Groq (Llama 3.3 70B)** replaces Gemini 2.5 Flash
- **Parallel fan-out** via async tool calls in a single LangGraph node
- **Structured output** via Pydantic + Groq JSON mode
- **Persistent memory** via SQLite + ChromaDB (semantic search)
- **Evaluation** via deterministic pre-check + LLM judge
- **Portfolio mode** with correlation synthesis
- **Output** via PDF (reportlab) and JSON

**Enhancements over v2 base:**
1. Deterministic hallucination pre-check (Python, before LLM judge)
2. Composite score trend delta (vs. last 3–5 runs)
3. MD&A text extraction from EDGAR 10-K filings
4. Groq semantic headline classification (replaces TextBlob)
5. Portfolio mode with parallel fan-out + correlation synthesis
6. ChromaDB vector memory for semantic search across reports

> ⚠️ All financial output is for informational purposes only. Not financial advice.

## 0 · Install Dependencies

In [ ]:
%%capture
!pip install langgraph langchain-groq langchain-core \
             yfinance httpx textblob reportlab \
             pydantic python-dotenv \
             chromadb sentence-transformers

## 1 · Configuration & Secrets

In [ ]:
import os

# ── Set your keys here or use Kaggle Secrets ──────────────────────────────
# In Kaggle: Add → Secrets → GROQ_API_KEY, GNEWS_API_KEY
# Then uncomment the lines below:
# from kaggle_secrets import UserSecretsClient
# secrets = UserSecretsClient()
# os.environ["GROQ_API_KEY"]  = secrets.get_secret("GROQ_API_KEY")
# os.environ["GNEWS_API_KEY"] = secrets.get_secret("GNEWS_API_KEY")  # free at gnews.io

# Or paste directly (not recommended for shared notebooks):
# os.environ["GROQ_API_KEY"]  = "gsk_..."
# os.environ["GNEWS_API_KEY"] = "..."

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
# GNEWS_API_KEY is read directly in get_news_sentiment() via os.environ.get()

# Model to use via Groq
GROQ_MODEL = "llama-3.3-70b-versatile"   # fast + capable; swap to llama3-8b-8192 if rate-limited

# Pipeline cost budget (USD) — pipeline aborts if exceeded
COST_BUDGET_USD = 0.05

# Judge score threshold: reports below this enter the refinement loop (max 2 iterations)
JUDGE_THRESHOLD = 4

# ChromaDB persistence directory
CHROMA_DB_PATH = "./chroma_finsight"

print("Config loaded. Model:", GROQ_MODEL)
print("GROQ key set:", bool(GROQ_API_KEY))
print("GNews key set:", bool(os.environ.get("GNEWS_API_KEY", "")))

## 2 · Pydantic Schemas

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import Optional, List, Union, Any, Dict


class RiskReport(BaseModel):
    """Final structured risk report — v2 enhanced schema."""
    ticker:              str            = Field(description="Stock ticker symbol.")
    risk_level:          str            = Field(description="'low', 'moderate', or 'high'.")
    volatility_pct:      float          = Field(description="52-week range as % of current price.")
    pe_ratio:            str            = Field(description="PE ratio as a string, e.g. '37.59' or 'N/A'.")
    summary:             str            = Field(description="One-sentence risk assessment.")
    recommendation:      str            = Field(description="'hold', 'watch', or 'avoid'.")
    implied_vol_pct:     Optional[float] = Field(default=None, description="ATM implied volatility annualised %, or null.")
    debt_to_equity:      Optional[float] = Field(default=None, description="Total debt / total equity from latest 10-K.")
    sentiment_score:     Optional[float] = Field(default=None, description="News sentiment -1 (bearish) to +1 (bullish), or null if unavailable.")
    dominant_theme:      Optional[str]   = Field(default=None, description="Dominant narrative theme from headlines.")
    market_regime:       Optional[str]   = Field(default=None, description="'risk-on', 'neutral', or 'risk-off'.")
    composite_score:     Optional[float] = Field(default=None, description="Weighted composite risk score 0-100.")
    data_quality:        str            = Field(description="'full', 'partial', or 'minimal'.")
    # New in enhanced version
    composite_trend:     Optional[str]  = Field(default=None, description="Trend delta string, e.g. '+12.4 pts in 7 days (rising risk)'.")
    mda_excerpt:         Optional[str]  = Field(default=None, description="Key excerpt from MD&A section of latest 10-K.")

    @field_validator('pe_ratio', mode='before')
    @classmethod
    def coerce_pe_ratio(cls, v):
        if v is None:
            return 'N/A'
        if isinstance(v, (int, float)):
            return str(round(float(v), 2))
        return str(v)

    @field_validator('sentiment_score', mode='before')
    @classmethod
    def coerce_sentiment(cls, v):
        if v is None or v == 'N/A' or v == '':
            return None
        try:
            return float(v)
        except (TypeError, ValueError):
            return None

    @field_validator('implied_vol_pct', mode='before')
    @classmethod
    def coerce_iv(cls, v):
        if v is None or v == 'N/A' or v == '':
            return None
        try:
            f = float(v)
            return f if f > 0 else None
        except (TypeError, ValueError):
            return None


class JudgeScore(BaseModel):
    score:     int  = Field(description="Quality score 1-5, where 5 is excellent.")
    reasoning: str  = Field(description="One sentence explaining the score.")


class HallucinationReport(BaseModel):
    has_hallucination:        bool       = Field(description="True if any numeric claim cannot be derived from tool outputs.")
    flagged_claims:           List[str]  = Field(description="List of claims that could not be verified.")
    confidence:               str        = Field(description="'high', 'medium', or 'low' confidence in the check.")
    deterministic_flags:      List[str]  = Field(default_factory=list, description="Fields flagged by deterministic pre-check.")
    deterministic_check_passed: bool     = Field(default=True, description="True if all deterministic checks passed.")


class BiasReport(BaseModel):
    sentiment_bias_detected: bool = Field(description="True if recommendation echoes headline sentiment over fundamentals.")
    reasoning:               str  = Field(description="One sentence explaining the bias assessment.")


class QuantData(BaseModel):
    rsi_signal:            str = Field(description="'overbought', 'oversold', or 'neutral'.")
    momentum_signal:       str = Field(description="'strong', 'weak', or 'neutral'.")
    macro_adjusted_signal: str = Field(description="Signal after macro overlay.")
    technical_summary:     str = Field(description="One-sentence technical outlook.")


class HeadlineClassification(BaseModel):
    """Groq-powered semantic classification for a single headline."""
    label:      str = Field(description="One of: catalyst_positive, catalyst_negative, noise, regulatory_risk.")
    confidence: str = Field(description="'high', 'medium', or 'low'.")
    reason:     str = Field(description="One-phrase rationale.")


class PortfolioSynthesis(BaseModel):
    """Cross-ticker synthesis for portfolio mode."""
    concentration_warnings: List[str] = Field(description="Sector/regime concentration alerts.")
    overall_portfolio_risk:  str       = Field(description="'low', 'moderate', or 'high'.")
    correlation_notes:       str       = Field(description="Key correlation observations.")
    portfolio_summary:       str       = Field(description="Two-sentence portfolio-level assessment.")


print("Schemas defined ✓")

## 3 · Tool Functions (Data Collection)

In [ ]:
import yfinance as yf
import httpx
from datetime import datetime, timedelta
import warnings, re, html
warnings.filterwarnings("ignore")


# ── Tool 1: Market data (yfinance) ────────────────────────────────────────
def get_market_data(ticker: str) -> dict:
    """Fetch real-time price, OHLCV, market cap, 52-week range."""
    try:
        stock = yf.Ticker(ticker.upper())
        info  = stock.info
        price = info.get("currentPrice") or info.get("regularMarketPrice")
        return {
            "status": "success",
            "data": {
                "ticker":          ticker.upper(),
                "current_price":   price,
                "52_week_high":    info.get("fiftyTwoWeekHigh"),
                "52_week_low":     info.get("fiftyTwoWeekLow"),
                "market_cap":      info.get("marketCap"),
                "pe_ratio":        info.get("trailingPE"),
                "volume":          info.get("volume"),
                "sector":          info.get("sector"),
                "company_name":    info.get("longName"),
            },
        }
    except Exception as e:
        return {"status": "error", "error_message": str(e)}


# ── Tool 2: Options implied volatility ───────────────────────────────────
def get_options_iv(ticker: str) -> dict:
    """Retrieve ATM implied volatility from the nearest-expiry options chain."""
    try:
        stock   = yf.Ticker(ticker.upper())
        expiries = stock.options
        if not expiries:
            return {"status": "error", "error_message": "No options data available."}
        nearest = expiries[0]
        chain   = stock.option_chain(nearest)
        calls   = chain.calls
        spot    = stock.info.get("currentPrice", 0)
        atm     = calls.iloc[(calls["strike"] - spot).abs().argsort()[:1]]
        iv      = float(atm["impliedVolatility"].values[0]) if not atm.empty else None
        return {
            "status": "success",
            "data": {
                "ticker":                  ticker.upper(),
                "expiry":                  nearest,
                "atm_implied_volatility":  iv,
                "iv_annualised_pct":       round(iv * 100, 2) if iv else None,
            },
        }
    except Exception as e:
        return {"status": "error", "error_message": str(e)}


# ── Tool 3: News sentiment — Groq semantic classifier ────────────────────
# ENHANCED: replaces TextBlob with a Groq call that classifies each headline
# as catalyst_positive, catalyst_negative, noise, or regulatory_risk.
# This produces a richer signal that feeds back into composite_score.

def classify_headline_groq(headline: str, llm_client) -> dict:
    """Classify a single headline using Groq. Returns HeadlineClassification dict."""
    from langchain_core.messages import HumanMessage, SystemMessage
    system = (
        "You are a financial headline classifier. "
        "Classify each headline as exactly one of: "
        "catalyst_positive, catalyst_negative, noise, regulatory_risk. "
        "catalyst_positive = concrete positive event (earnings beat, deal, upgrade). "
        "catalyst_negative = concrete negative event (miss, downgrade, legal loss). "
        "regulatory_risk   = regulatory, legal, or compliance risk. "
        "noise              = generic mentions with no directional signal. "
        "Respond ONLY with valid JSON: {\"label\": str, \"confidence\": str, \"reason\": str}"
    )
    messages = [
        SystemMessage(content=system),
        HumanMessage(content=f"Headline: {headline}"),
    ]
    try:
        response = llm_client.invoke(messages)
        raw = response.content.strip()
        raw = re.sub(r"^```(?:json)?\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
        parsed = json.loads(raw)
        return HeadlineClassification(**parsed).model_dump()
    except Exception:
        return {"label": "noise", "confidence": "low", "reason": "parse error"}


LABEL_SCORES = {
    "catalyst_positive": +1.0,
    "catalyst_negative": -1.0,
    "regulatory_risk":   -0.6,
    "noise":              0.0,
}

def get_news_sentiment(ticker: str, days_back: int = 7, llm_client=None) -> dict:
    """
    Fetch headlines from GNews API and classify with Groq (or fall back to
    TextBlob if llm_client is None or GNEWS_API_KEY is absent).
    """
    gnews_key = os.environ.get("GNEWS_API_KEY", "")
    if not gnews_key:
        return {"status": "skipped", "error_message": "GNEWS_API_KEY not set — sentiment skipped."}
    try:
        from_date = (datetime.utcnow() - timedelta(days=days_back)).strftime("%Y-%m-%dT%H:%M:%SZ")
        url = (
            f"https://gnews.io/api/v4/search?"
            f"q={ticker}&lang=en&max=10"
            f"&from={from_date}"
            f"&apikey={gnews_key}"
        )
        resp = httpx.get(url, timeout=10)
        resp.raise_for_status()
        articles  = resp.json().get("articles", [])
        headlines = [a["title"] for a in articles if a.get("title")]
        label_counts = {}

        if llm_client and headlines:
            # Groq semantic classification — richer signal
            classifications = [classify_headline_groq(h, llm_client) for h in headlines]
            scores = [LABEL_SCORES.get(c["label"], 0.0) for c in classifications]
            label_counts = {}
            for c in classifications:
                label_counts[c["label"]] = label_counts.get(c["label"], 0) + 1
            dominant_label = max(label_counts, key=label_counts.get) if label_counts else "noise"
            method = "groq_semantic"
        else:
            # Fallback: TextBlob
            from textblob import TextBlob
            classifications = [{"label": "noise", "confidence": "low", "reason": "textblob"} for _ in headlines]
            scores = [TextBlob(h).sentiment.polarity for h in headlines]
            dominant_label = "noise"
            method = "textblob_fallback"

        avg_score = sum(scores) / len(scores) if scores else 0.0
        return {
            "status": "success",
            "data": {
                "ticker":           ticker,
                "headline_count":   len(headlines),
                "headlines":        headlines[:10],
                "classifications":  classifications[:10],
                "sentiment_score":  round(avg_score, 4),
                "dominant_label":   dominant_label,
                "label_counts":     label_counts if llm_client else {},
                "sentiment_label":  (
                    "positive" if avg_score > 0.1
                    else "negative" if avg_score < -0.1
                    else "neutral"
                ),
                "method":           method,
                "source_count": len({a.get("source", {}).get("name") for a in articles}),
            },
        }
    except Exception as e:
        return {"status": "error", "error_message": str(e)}


# ── Tool 4: SEC EDGAR filings + MD&A extraction ───────────────────────────
# ENHANCED: resolves ticker -> CIK, reads SEC submissions JSON, finds the
# latest 10-K accession, fetches the archive index JSON, then slices MD&A
# from the actual primary filing document.

SEC_HEADERS = {
    "User-Agent": "FinSight research@finsight.dev",
    "Accept-Encoding": "gzip, deflate",
}

MDA_PATTERNS = [
    r"(?is)item\s+7\.?\s*management['’`s\s]+discussion\s+and\s+analysis.*?(?=item\s+7a\.?|item\s+8\.?|\Z)",
    r"(?is)management['’`s\s]+discussion\s+and\s+analysis\s+of\s+financial\s+condition.*?(?=quantitative\s+and\s+qualitative|item\s+8\.?|\Z)",
]


def _sec_get_json(url: str, timeout: int = 15) -> dict:
    resp = httpx.get(url, headers=SEC_HEADERS, timeout=timeout, follow_redirects=True)
    resp.raise_for_status()
    return resp.json()


def _sec_get_text(url: str, timeout: int = 20) -> str:
    resp = httpx.get(url, headers=SEC_HEADERS, timeout=timeout, follow_redirects=True)
    resp.raise_for_status()
    return resp.text


def resolve_cik_for_ticker(ticker: str) -> str:
    """Resolve ticker to a zero-padded 10-digit SEC CIK."""
    ticker = ticker.upper().replace(".", "-")
    data = _sec_get_json("https://www.sec.gov/files/company_tickers.json")
    for row in data.values():
        if row.get("ticker", "").upper() == ticker:
            return str(row["cik_str"]).zfill(10)
    raise ValueError(f"Ticker {ticker} not found in SEC company_tickers.json")


def find_latest_filing(submissions: dict, form_type: str = "10-K") -> dict:
    """Return metadata for the most recent filing of form_type from submissions JSON."""
    recent = submissions.get("filings", {}).get("recent", {})
    forms = recent.get("form", [])
    for idx, form in enumerate(forms):
        if form == form_type:
            return {
                "accession": recent.get("accessionNumber", [])[idx],
                "primary_document": recent.get("primaryDocument", [None])[idx],
                "filing_date": recent.get("filingDate", ["unknown"])[idx],
                "report_date": recent.get("reportDate", ["unknown"])[idx],
                "form_type": form,
            }
    raise ValueError(f"No recent {form_type} filing found")


def get_primary_document_from_index(cik: str, accession: str, preferred: str = None) -> str:
    """Fetch SEC archive index JSON and return the primary filing document filename."""
    cik_int = str(int(cik))
    accession_clean = accession.replace("-", "")
    index_url = f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{accession_clean}/{accession}-index.json"
    index_json = _sec_get_json(index_url)
    items = index_json.get("directory", {}).get("item", [])
    names = [item.get("name") for item in items if item.get("name")]

    if preferred and preferred in names:
        return preferred
    for name in names:
        lower = name.lower()
        if lower.endswith((".htm", ".html", ".txt")) and not lower.endswith("-index.html"):
            if "exhibit" not in lower and not lower.startswith("ex"):
                return name
    raise ValueError("Could not identify primary SEC filing document from index JSON")


def _plain_text_from_html(raw: str) -> str:
    raw = re.sub(r"(?is)<(script|style).*?>.*?</\1>", " ", raw)
    raw = re.sub(r"(?is)<br\s*/?>", "\n", raw)
    raw = re.sub(r"(?is)</p>|</div>|</tr>|</h[1-6]>", "\n", raw)
    text = re.sub(r"<[^>]+>", " ", raw)
    text = html.unescape(text)
    text = re.sub(r"\xa0", " ", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_mda_from_document(document_url: str, max_chars: int = 4000) -> str:
    """Fetch a 10-K document URL and extract the MD&A section."""
    try:
        text = _plain_text_from_html(_sec_get_text(document_url))
        normalized = re.sub(r"\s+", " ", text)
        for pattern in MDA_PATTERNS:
            match = re.search(pattern, normalized)
            if match:
                return match.group(0)[:max_chars].strip()
        idx = normalized.lower().find("management's discussion and analysis")
        if idx < 0:
            idx = normalized.lower().find("management’s discussion and analysis")
        return normalized[idx: idx + max_chars].strip() if idx >= 0 else ""
    except Exception as e:
        return f"[MD&A extraction failed: {e}]"


def get_sec_filings(ticker: str, form_type: str = "10-K", extract_mda: bool = True) -> dict:
    """Fetch most recent SEC filing metadata + optional MD&A text for a ticker."""
    try:
        cik_padded = resolve_cik_for_ticker(ticker)
        cik_int = str(int(cik_padded))
        submissions_url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"
        submissions = _sec_get_json(submissions_url)
        filing = find_latest_filing(submissions, form_type)

        accession = filing["accession"]
        accession_clean = accession.replace("-", "")
        primary_doc = get_primary_document_from_index(cik_padded, accession, filing.get("primary_document"))
        document_url = f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{accession_clean}/{primary_doc}"
        mda_text = extract_mda_from_document(document_url) if extract_mda else ""

        # Debt/equity from yfinance balance sheet
        stock        = yf.Ticker(ticker.upper())
        info         = stock.info
        total_debt   = info.get("totalDebt", 0) or 0
        total_equity = info.get("bookValue", 1) * info.get("sharesOutstanding", 1) or 1
        roe          = info.get("returnOnEquity")
        return {
            "status": "success",
            "data": {
                "ticker":           ticker.upper(),
                "cik":              cik_padded,
                "form_type":        form_type,
                "entity_name":      submissions.get("name", ticker),
                "accession_number": accession,
                "primary_document": primary_doc,
                "filing_url":       document_url,
                "period_of_report": filing.get("report_date", "unknown"),
                "file_date":        filing.get("filing_date", "unknown"),
                "total_debt":       total_debt,
                "total_equity":     total_equity,
                "debt_to_equity":   round(total_debt / total_equity, 4) if total_equity else None,
                "return_on_equity": roe,
                "mda_excerpt":      mda_text[:2000] if mda_text else None,
            },
        }
    except Exception as e:
        return {"status": "error", "error_message": str(e)}


# ── Tool 5: Macro context ─────────────────────────────────────────────────
def get_macro_context() -> dict:
    """Retrieve VIX, 10Y Treasury yield, S&P 500 30-day return, and market regime."""
    try:
        vix   = yf.Ticker("^VIX").history(period="5d")["Close"].iloc[-1]
        tnx   = yf.Ticker("^TNX").history(period="5d")["Close"].iloc[-1]
        spx   = yf.Ticker("^GSPC").history(period="35d")["Close"]
        sp_ret = (spx.iloc[-1] / spx.iloc[0] - 1) * 100
        regime = (
            "risk-off" if vix > 25 else
            "neutral"  if vix > 15 else
            "risk-on"
        )
        return {
            "status": "success",
            "data": {
                "vix":                  round(float(vix), 2),
                "ten_year_yield_pct":   round(float(tnx), 2),
                "sp500_30d_return_pct": round(float(sp_ret), 2),
                "market_regime":        regime,
            },
        }
    except Exception as e:
        return {"status": "error", "error_message": str(e)}


print("Tool functions defined ✓")

## 4 · Composite Risk Score & Guardrails

In [ ]:
# ── Composite risk score (Section 5.2 formula) ───────────────────────────
def compute_composite_score(
    volatility_pct:  float,
    implied_vol_pct: Optional[float],
    debt_to_equity:  Optional[float],
    sentiment_score: Optional[float],
    market_regime:   Optional[str],
) -> float:
    """
    Weighted composite risk score 0–100 (100 = maximum risk).
    Weights: hist_vol=0.30, impl_vol=0.25, leverage=0.20, sentiment=0.15, macro=0.10
    """
    w1, w2, w3, w4, w5 = 0.30, 0.25, 0.20, 0.15, 0.10

    v_hist = min(volatility_pct / 2.0, 100)
    v_impl = min(implied_vol_pct or 30.0, 100)
    d_lev  = min((debt_to_equity or 1.0), 5) / 5 * 100
    s_sent = (-(sentiment_score or 0.0)) * 50 + 50
    m_macro = {"risk-off": 20, "neutral": 10, "risk-on": 0}.get(market_regime or "neutral", 10)

    score = w1*v_hist + w2*v_impl + w3*d_lev + w4*s_sent + w5*m_macro
    return round(min(score, 100), 2)


# ── ENHANCED: Deterministic hallucination pre-check ──────────────────────
# Runs BEFORE the LLM judge. Recomputes key numerics from raw tool outputs
# and asserts they match the report within tolerance. The LLM judge then
# only handles qualitative claims it cannot check mechanically.

TOLERANCE_PCT = 0.02   # 2% relative tolerance for float comparisons

def deterministic_hallucination_check(
    risk_report: dict,
    market_data: dict,
    iv_data: dict,
    fundamentals_data: dict,
    news_sentiment_data: dict,
    macro_data: dict,
) -> dict:
    """
    Recompute volatility_pct and composite_score from raw tool outputs.
    Assert they match the report within TOLERANCE_PCT.
    Returns a dict with deterministic_check_passed and deterministic_flags.
    """
    flags = []

    md   = market_data.get("data", {})
    iv   = iv_data.get("data", {})
    fund = fundamentals_data.get("data", {})
    news = news_sentiment_data.get("data", {})
    mac  = macro_data.get("data", {})

    # Recompute volatility_pct
    price = md.get("current_price") or 1
    hi    = md.get("52_week_high", price)
    lo    = md.get("52_week_low", price)
    expected_vol = round((hi - lo) / price * 100, 2) if price else None

    reported_vol = risk_report.get("volatility_pct")
    if expected_vol is not None and reported_vol is not None:
        rel_err = abs(reported_vol - expected_vol) / (abs(expected_vol) + 1e-9)
        if rel_err > TOLERANCE_PCT:
            flags.append(
                f"volatility_pct mismatch: report={reported_vol}, "
                f"recomputed={expected_vol} (err={rel_err:.1%})"
            )

    # Recompute composite_score
    iv_pct   = iv.get("iv_annualised_pct")
    d_e      = fund.get("debt_to_equity")
    sent     = news.get("sentiment_score")
    regime   = mac.get("market_regime")
    vol_for_composite = expected_vol if expected_vol is not None else (reported_vol or 0)
    expected_composite = compute_composite_score(vol_for_composite, iv_pct, d_e, sent, regime)

    reported_composite = risk_report.get("composite_score")
    if reported_composite is not None:
        rel_err = abs(reported_composite - expected_composite) / (abs(expected_composite) + 1e-9)
        if rel_err > TOLERANCE_PCT:
            flags.append(
                f"composite_score mismatch: report={reported_composite}, "
                f"recomputed={expected_composite} (err={rel_err:.1%})"
            )

    # Check sentiment_score: must match tool output within tolerance
    reported_sent = risk_report.get("sentiment_score")
    tool_sent     = news.get("sentiment_score")
    if tool_sent is not None and reported_sent is not None:
        rel_err = abs(reported_sent - tool_sent) / (abs(tool_sent) + 1e-9)
        if rel_err > TOLERANCE_PCT:
            flags.append(
                f"sentiment_score mismatch: report={reported_sent}, "
                f"tool={tool_sent} (err={rel_err:.1%})"
            )

    # Check debt_to_equity
    reported_d_e = risk_report.get("debt_to_equity")
    if d_e is not None and reported_d_e is not None:
        rel_err = abs(reported_d_e - d_e) / (abs(d_e) + 1e-9)
        if rel_err > TOLERANCE_PCT:
            flags.append(
                f"debt_to_equity mismatch: report={reported_d_e}, "
                f"tool={d_e} (err={rel_err:.1%})"
            )

    passed = len(flags) == 0
    if not passed:
        print(f"  ⚠ Deterministic pre-check FAILED: {flags}")
    else:
        print("  ✓ Deterministic pre-check passed — all numerics verified")

    return {"deterministic_check_passed": passed, "deterministic_flags": flags}


# ── Compliance filter ─────────────────────────────────────────────────────
DISCLAIMER = (
    "\n\n---\n"
    "DISCLAIMER: This report is generated by FinSight AI for informational purposes only. "
    "It does not constitute financial advice, investment advice, or a recommendation to buy, "
    "hold, or sell any security. Always consult a qualified financial adviser before making "
    "investment decisions. Past performance is not indicative of future results.\n"
)

PROHIBITED_CLAIMS = [
    "guaranteed return", "risk-free", "certain to", "will definitely"
]

def apply_compliance_filter(report_text: str) -> tuple:
    redacted = []
    for phrase in PROHIBITED_CLAIMS:
        if phrase.lower() in report_text.lower():
            report_text = report_text.replace(phrase, "[REDACTED]")
            redacted.append(phrase)
    return report_text + DISCLAIMER, redacted


# ── Cost guardrail ────────────────────────────────────────────────────────
_cost_state = {"total_usd": 0.0, "input_tokens": 0, "output_tokens": 0, "budget_exceeded": False, "budget_message": ""}

COST_PER_1K_INPUT  = 0.00059
COST_PER_1K_OUTPUT = 0.00079

def reset_cost_state() -> None:
    _cost_state.update({
        "total_usd": 0.0,
        "input_tokens": 0,
        "output_tokens": 0,
        "budget_exceeded": False,
        "budget_message": "",
    })


def record_cost(input_tokens: int, output_tokens: int) -> None:
    _cost_state["input_tokens"]  += input_tokens
    _cost_state["output_tokens"] += output_tokens
    _cost_state["total_usd"] += (
        input_tokens  / 1000 * COST_PER_1K_INPUT +
        output_tokens / 1000 * COST_PER_1K_OUTPUT
    )
    if _cost_state["total_usd"] > COST_BUDGET_USD and not _cost_state.get("budget_exceeded"):
        _cost_state["budget_exceeded"] = True
        _cost_state["budget_message"] = (
            f"Pipeline budget exceeded: ${_cost_state['total_usd']:.4f} > ${COST_BUDGET_USD:.4f}. "
            "Finalizing with partial results."
        )
        print(f"  ⚠ {_cost_state['budget_message']}")


def budget_exceeded() -> bool:
    return bool(_cost_state.get("budget_exceeded"))


def cost_summary() -> dict:
    return {
        "total_cost_usd":     round(_cost_state["total_usd"], 6),
        "total_input_tokens": _cost_state["input_tokens"],
        "total_output_tokens":_cost_state["output_tokens"],
        "budget_exceeded":    _cost_state.get("budget_exceeded", False),
        "budget_message":     _cost_state.get("budget_message", ""),
    }


print("Composite score formula, deterministic pre-check, compliance filter, cost guardrail defined ✓")

## 5 · SQLite Persistence

In [ ]:
import sqlite3, json
from pathlib import Path

DB_PATH = Path("finsight.db")

def init_db() -> None:
    with sqlite3.connect(DB_PATH) as conn:
        conn.executescript("""
            CREATE TABLE IF NOT EXISTS reports (
                id          INTEGER PRIMARY KEY AUTOINCREMENT,
                ticker      TEXT NOT NULL,
                run_at      TEXT NOT NULL,
                risk_report TEXT NOT NULL,
                judge_score TEXT NOT NULL,
                composite   REAL,
                token_cost  INTEGER
            );
            CREATE INDEX IF NOT EXISTS idx_reports_ticker ON reports(ticker);
        """)

def save_report(ticker: str, risk_report: dict, judge_score: dict, token_cost: int = 0) -> int:
    with sqlite3.connect(DB_PATH) as conn:
        cur = conn.execute(
            "INSERT INTO reports (ticker, run_at, risk_report, judge_score, composite, token_cost) "
            "VALUES (?,?,?,?,?,?)",
            (
                ticker,
                datetime.utcnow().isoformat(),
                json.dumps(risk_report),
                json.dumps(judge_score),
                risk_report.get("composite_score"),
                token_cost,
            ),
        )
        return cur.lastrowid

def get_report_history(ticker: str, limit: int = 5) -> list:
    with sqlite3.connect(DB_PATH) as conn:
        rows = conn.execute(
            "SELECT run_at, risk_report, judge_score, composite FROM reports "
            "WHERE ticker=? ORDER BY run_at DESC LIMIT ?",
            (ticker, limit),
        ).fetchall()
    return [
        {"run_at": r[0], "risk_report": json.loads(r[1]),
         "judge_score": json.loads(r[2]), "composite": r[3]}
        for r in rows
    ]


# ── ENHANCED: Composite score trend delta ─────────────────────────────────
# Compares the current composite score against the last 3–5 runs stored in
# SQLite and emits a delta signal. One extra function against data we already
# have; makes the recommendation dramatically more actionable.

def compute_composite_trend(ticker: str, current_composite: float, lookback: int = 5) -> str:
    """
    Compare current_composite against the last `lookback` runs for this ticker.
    Returns a human-readable trend string, e.g.:
      '+12.4 pts in 7 days (rising risk)'
      '-5.1 pts in 14 days (improving)'
      'No prior data'
    """
    history = get_report_history(ticker, limit=lookback)
    if not history:
        return "No prior data — first run for this ticker."

    # Most recent prior run (history is DESC, so history[0] is most recent saved)
    prior = history[0]
    prior_composite = prior.get("composite")
    if prior_composite is None:
        return "Prior run had no composite score."

    try:
        prior_dt  = datetime.fromisoformat(prior["run_at"])
        now_dt    = datetime.utcnow()
        days_diff = max((now_dt - prior_dt).days, 1)
    except Exception:
        days_diff = 0

    delta = round(current_composite - prior_composite, 2)
    direction = "rising risk" if delta > 0 else "improving" if delta < 0 else "stable"
    sign = "+" if delta >= 0 else ""

    # Multi-run average for longer trend context
    if len(history) >= 3:
        oldest = history[-1]
        oldest_composite = oldest.get("composite")
        if oldest_composite is not None:
            try:
                oldest_dt = datetime.fromisoformat(oldest["run_at"])
                span_days = max((now_dt - oldest_dt).days, 1)
            except Exception:
                span_days = 0
            long_delta = round(current_composite - oldest_composite, 2)
            long_sign  = "+" if long_delta >= 0 else ""
            return (
                f"{sign}{delta} pts vs last run ({days_diff}d ago, {direction}); "
                f"{long_sign}{long_delta} pts over {len(history)} runs / {span_days}d"
            )

    return f"{sign}{delta} pts in {days_diff} days ({direction})"


init_db()
print("SQLite database initialised ✓")

## 6 · ChromaDB Vector Memory

In [ ]:
# ── ENHANCED: ChromaDB vector memory ─────────────────────────────────────
# Persists report summaries as embeddings for semantic search.
# Useful once you have hundreds of reports and want to query:
#   "find all reports where dominant theme was regulatory_risk"
#   "show me tickers similar to NVDA's current risk profile"

import chromadb
from chromadb.utils import embedding_functions

_chroma_client = None
_chroma_collection = None

def get_chroma_collection():
    global _chroma_client, _chroma_collection
    if _chroma_collection is not None:
        return _chroma_collection
    try:
        _chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
        ef = embedding_functions.DefaultEmbeddingFunction()  # uses all-MiniLM-L6-v2
        _chroma_collection = _chroma_client.get_or_create_collection(
            name="finsight_reports",
            embedding_function=ef,
            metadata={"hnsw:space": "cosine"},
        )
        print(f"  ChromaDB collection ready: {_chroma_collection.count()} existing docs")
        return _chroma_collection
    except Exception as e:
        print(f"  ⚠ ChromaDB unavailable: {e}")
        return None


def chroma_store_report(risk_report: dict, judge_score: dict, run_at: str = None) -> None:
    """
    Store a risk report summary in ChromaDB for future semantic search.
    The document text is a rich description of the report; metadata holds
    structured fields for filtered retrieval.
    """
    collection = get_chroma_collection()
    if collection is None:
        return
    ticker    = risk_report.get("ticker", "UNKNOWN")
    run_ts    = run_at or datetime.utcnow().isoformat()
    doc_id    = f"{ticker}_{run_ts.replace(':', '-').replace('.', '-')}"

    # Build a rich text description for embedding
    doc_text = (
        f"Ticker: {ticker}. "
        f"Risk level: {risk_report.get('risk_level', 'N/A')}. "
        f"Composite score: {risk_report.get('composite_score', 'N/A')}. "
        f"Summary: {risk_report.get('summary', '')}. "
        f"Dominant theme: {risk_report.get('dominant_theme', 'N/A')}. "
        f"Market regime: {risk_report.get('market_regime', 'N/A')}. "
        f"Recommendation: {risk_report.get('recommendation', 'N/A')}. "
        f"Sentiment score: {risk_report.get('sentiment_score', 'N/A')}. "
        f"Composite trend: {risk_report.get('composite_trend', 'N/A')}. "
        f"Judge score: {judge_score.get('score', 'N/A')}/5. "
        f"Judge reasoning: {judge_score.get('reasoning', '')}."
    )

    metadata = {
        "ticker":          ticker,
        "risk_level":      risk_report.get("risk_level", ""),
        "composite_score": float(risk_report.get("composite_score") or 0),
        "recommendation":  risk_report.get("recommendation", ""),
        "market_regime":   risk_report.get("market_regime", ""),
        "judge_score":     int(judge_score.get("score") or 0),
        "run_at":          run_ts,
    }
    try:
        collection.add(documents=[doc_text], metadatas=[metadata], ids=[doc_id])
        print(f"  ✓ ChromaDB: stored report for {ticker} (id={doc_id})")
    except Exception as e:
        print(f"  ⚠ ChromaDB store error: {e}")


def chroma_search(query: str, n_results: int = 5, filter_metadata: dict = None) -> list:
    """
    Semantic search across all stored reports.
    Example: chroma_search("regulatory risk semiconductor", n_results=5)
             chroma_search("high risk avoid", filter_metadata={"market_regime": "risk-off"})
    """
    collection = get_chroma_collection()
    if collection is None:
        return []
    try:
        kwargs = {"query_texts": [query], "n_results": min(n_results, max(collection.count(), 1))}
        if filter_metadata:
            kwargs["where"] = filter_metadata
        results = collection.query(**kwargs)
        out = []
        for i, doc in enumerate(results["documents"][0]):
            out.append({
                "document":  doc,
                "metadata":  results["metadatas"][0][i],
                "distance":  results["distances"][0][i],
            })
        return out
    except Exception as e:
        print(f"  ⚠ ChromaDB search error: {e}")
        return []


print("ChromaDB vector memory defined ✓")

## 7 · LangGraph State & LLM Helper

In [ ]:
import re, time
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage


# ── Pipeline state ────────────────────────────────────────────────────────
class FinSightState(TypedDict):
    ticker:               str
    market_data:          dict
    iv_data:              dict
    fundamentals_data:    dict
    news_sentiment_data:  dict
    macro_data:           dict
    quant_data:           dict
    risk_report:          dict
    judge_score:          dict
    hallucination_report: dict
    bias_report:          dict
    refinement_iter:      int
    final_report_text:    str
    error:                str


# ── Groq LLM client ───────────────────────────────────────────────────────
llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=GROQ_API_KEY,
    temperature=0.1,
)


def call_llm_json(system: str, user: str, schema_class) -> dict:
    """Call Groq and parse the response as JSON matching schema_class."""
    if budget_exceeded():
        print(f"  ⚠ Skipping {schema_class.__name__} LLM call: {_cost_state.get('budget_message')}")
        return {}
    messages = [
        SystemMessage(content=system + "\n\nRespond ONLY with valid JSON. No markdown fences, no explanation."),
        HumanMessage(content=user),
    ]
    response = llm.invoke(messages)
    usage = getattr(response, "usage_metadata", None)
    if usage:
        record_cost(
            usage.get("input_tokens", 0),
            usage.get("output_tokens", 0),
        )
    raw = response.content.strip()
    raw = re.sub(r"^```(?:json)?\n?", "", raw)
    raw = re.sub(r"\n?```$", "", raw)
    try:
        parsed = json.loads(raw)
        return schema_class(**parsed).model_dump()
    except Exception as e:
        print(f"  ⚠ JSON parse error ({schema_class.__name__}): {e}\nRaw: {raw[:300]}")
        return {}


print("LangGraph state and LLM client defined ✓")

## 8 · LangGraph Node Functions

In [ ]:
import concurrent.futures


# ── Node 1: Parallel data collection ─────────────────────────────────────
def node_collect_data(state: FinSightState) -> FinSightState:
    ticker = state["ticker"]
    print(f"  [collect_data] Fetching data for {ticker} in parallel...")
    t0 = time.time()
    results = {}
    errors = []

    tasks = {
        "market_data":         (get_market_data, (ticker,)),
        "iv_data":             (get_options_iv, (ticker,)),
        "fundamentals_data":   (get_sec_filings, (ticker, "10-K", True)),
        "news_sentiment_data": (get_news_sentiment, (ticker, 7, llm)),
        "macro_data":          (get_macro_context, ()),
    }
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as ex:
        futures = {ex.submit(fn, *args): name for name, (fn, args) in tasks.items()}
        for future in concurrent.futures.as_completed(futures):
            name = futures[future]
            try:
                results[name] = future.result()
            except Exception as exc:
                msg = f"{name} failed: {exc}"
                errors.append(msg)
                results[name] = {"status": "error", "error_message": str(exc)}

    print(f"  [collect_data] Done in {time.time()-t0:.1f}s")
    for k, v in results.items():
        print(f"    {k}: {v.get('status', 'unknown')}")

    error_text = "; ".join(filter(None, [state.get("error", ""), *errors]))
    return {**state, **results, "error": error_text}


# ── Node 2: Quant strategy signals ────────────────────────────────────────
def node_quant_strategy(state: FinSightState) -> FinSightState:
    if budget_exceeded():
        return {**state, "error": _cost_state.get("budget_message", state.get("error", ""))}
    print("  [quant_strategy] Computing RSI / momentum / macro overlay...")
    md  = state["market_data"].get("data", {})
    mac = state["macro_data"].get("data", {})

    system = "You are a quantitative analyst. Compute technical signals from the data provided."
    user = f"""
Market data: {json.dumps(md)}
Macro data:  {json.dumps(mac)}

Compute:
1. 14-day RSI classification using 52-week high/low and current price as proxy:
   - If current_price > 0.90 * 52_week_high → 'overbought'
   - If current_price < 1.10 * 52_week_low  → 'oversold'
   - Otherwise → 'neutral'
2. Momentum signal:
   - 'strong' if current_price > 0.95 * 52_week_high
   - 'weak'   if current_price < 1.10 * 52_week_low
   - 'neutral' otherwise
3. Macro-adjusted signal: shift the momentum signal one level toward risk in the direction
   of market_regime ('risk-off' worsens, 'risk-on' improves).
4. technical_summary: one sentence combining all three signals.

Return JSON matching: {{rsi_signal, momentum_signal, macro_adjusted_signal, technical_summary}}
"""
    result = call_llm_json(system, user, QuantData)
    return {**state, "quant_data": result}


# ── Node 3: Risk synthesis (with MD&A grounding) ──────────────────────────
# ENHANCED: MD&A excerpt is included in the synthesis prompt so the LLM
# grounds qualitative statements in actual document language.
def node_risk_synthesis(state: FinSightState) -> FinSightState:
    if budget_exceeded():
        return {**state, "error": _cost_state.get("budget_message", state.get("error", ""))}
    print("  [risk_synthesis] Synthesising risk report...")
    md   = state["market_data"].get("data", {})
    iv   = state["iv_data"].get("data", {})
    fund = state["fundamentals_data"].get("data", {})
    news = state["news_sentiment_data"].get("data", {})
    mac  = state["macro_data"].get("data", {})
    qnt  = state.get("quant_data", {})
    ticker = state["ticker"]

    # Pre-compute composite score deterministically
    price    = md.get("current_price", 0) or 1
    hi       = md.get("52_week_high", price)
    lo       = md.get("52_week_low", price)
    vol_pct  = round((hi - lo) / price * 100, 2)
    iv_pct   = iv.get("iv_annualised_pct")
    d_e      = fund.get("debt_to_equity")
    sent     = news.get("sentiment_score")
    regime   = mac.get("market_regime")
    composite = compute_composite_score(vol_pct, iv_pct, d_e, sent, regime)

    # Compute trend delta against SQLite history
    composite_trend = compute_composite_trend(ticker, composite)
    print(f"  [risk_synthesis] Composite trend: {composite_trend}")

    # MD&A excerpt from EDGAR (if available)
    mda_excerpt = fund.get("mda_excerpt", "")
    mda_section = ""
    if mda_excerpt and not mda_excerpt.startswith("[MD&A"):
        mda_section = f"""
Management Discussion & Analysis (MD&A) excerpt from latest 10-K:
--- BEGIN MD&A ---
{mda_excerpt[:1500]}
--- END MD&A ---
Use language from the MD&A to ground qualitative risk statements where relevant.
"""

    available = sum([
        state["market_data"].get("status") == "success",
        state["iv_data"].get("status") == "success",
        state["fundamentals_data"].get("status") == "success",
        state["news_sentiment_data"].get("status") in ("success", "skipped"),
        state["macro_data"].get("status") == "success",
    ])
    data_quality = "full" if available == 5 else "partial" if available >= 3 else "minimal"

    # Include Groq headline classifications in synthesis prompt
    headline_summary = ""
    if news.get("label_counts"):
        headline_summary = f"Headline label counts: {json.dumps(news.get('label_counts', {}))}. Dominant label: {news.get('dominant_label', 'N/A')}."

    system = "You are a senior financial risk analyst. Produce a structured risk report in JSON."
    user = f"""
Ticker: {ticker}
Market data:     {json.dumps(md)}
Options IV:      {json.dumps(iv)}
Fundamentals:    {json.dumps(fund)}
News sentiment:  {json.dumps({k:v for k,v in news.items() if k != 'classifications'})}
{headline_summary}
Macro context:   {json.dumps(mac)}
Quant signals:   {json.dumps(qnt)}
{mda_section}
Pre-computed values (use these exactly):
  volatility_pct:  {vol_pct}
  composite_score: {composite}  (0-100, >60=high, 30-60=moderate, <30=low)
  composite_trend: {composite_trend}
  data_quality:    {data_quality}

Based on ALL data above, produce a RiskReport JSON with these fields:
  ticker, risk_level (low/moderate/high), volatility_pct, pe_ratio (number or 'N/A'),
  summary (one sentence, ground in MD&A language where available),
  recommendation (hold/watch/avoid),
  implied_vol_pct, debt_to_equity, sentiment_score, dominant_theme,
  market_regime, composite_score, data_quality,
  composite_trend (copy the pre-computed string exactly),
  mda_excerpt (key phrase from MD&A that supports your summary, or null)

Use the pre-computed volatility_pct and composite_score exactly as given.
Set risk_level based on composite_score: <30=low, 30-60=moderate, >60=high.
"""
    result = call_llm_json(system, user, RiskReport)
    # Ensure trend is always set from our deterministic computation
    if result:
        result["composite_trend"] = composite_trend
    return {**state, "risk_report": result}


# ── Node 4: Judge ─────────────────────────────────────────────────────────
def node_judge(state: FinSightState) -> FinSightState:
    if budget_exceeded():
        return {**state, "judge_score": {"score": 0, "reasoning": _cost_state.get("budget_message", "Budget exceeded")}, "error": _cost_state.get("budget_message", state.get("error", ""))}
    print("  [judge] Scoring report quality...")
    system = "You are a quality judge for AI-generated financial reports."
    user = f"""
Score this risk report on a scale of 1-5 (5=excellent).
Criteria: completeness, internal consistency, actionability, grounding in data.

Report: {json.dumps(state['risk_report'])}

Return JSON: {{"score": <int 1-5>, "reasoning": "<one sentence>"}}
"""
    result = call_llm_json(system, user, JudgeScore)
    print(f"  [judge] Score: {result.get('score')}/5 — {result.get('reasoning')}")
    return {**state, "judge_score": result}


# ── Node 5: Hallucination check (deterministic + LLM) ─────────────────────
# ENHANCED: deterministic Python pre-check runs first, then LLM judge only
# handles qualitative claims it cannot verify mechanically.
def node_hallucination_check(state: FinSightState) -> FinSightState:
    if budget_exceeded():
        return {**state, "error": _cost_state.get("budget_message", state.get("error", ""))}
    print("  [hallucination_check] Running deterministic pre-check...")

    # Step 1: Deterministic check
    det = deterministic_hallucination_check(
        risk_report          = state["risk_report"],
        market_data          = state["market_data"],
        iv_data              = state["iv_data"],
        fundamentals_data    = state["fundamentals_data"],
        news_sentiment_data  = state["news_sentiment_data"],
        macro_data           = state["macro_data"],
    )

    # Step 2: LLM judge for qualitative claims only (numerics already verified)
    print("  [hallucination_check] LLM qualitative check...")
    system = "You are a fact-checker for AI-generated financial reports. Focus ONLY on qualitative claims."
    user = f"""
Risk report: {json.dumps(state['risk_report'])}

Ground truth tool outputs:
  market_data:       {json.dumps(state['market_data'].get('data', {}))}
  fundamentals_data: {json.dumps(state['fundamentals_data'].get('data', {}))}
  news_sentiment:    {json.dumps({k:v for k,v in state['news_sentiment_data'].get('data', {}).items() if k != 'classifications'})}
  macro_data:        {json.dumps(state['macro_data'].get('data', {}))}

The numeric fields (volatility_pct, composite_score, sentiment_score, debt_to_equity)
have already been verified deterministically. Focus ONLY on:
- The qualitative 'summary' text: does it contradict the data?
- The 'dominant_theme': is it supported by the headlines?
- The 'mda_excerpt': is it plausible given the filing metadata?

Return JSON: {{
  "has_hallucination": bool,
  "flagged_claims": [list of strings],
  "confidence": "high"|"medium"|"low",
  "deterministic_flags": [],
  "deterministic_check_passed": true
}}
"""
    llm_result = call_llm_json(system, user, HallucinationReport)

    # Merge deterministic and LLM results
    combined = {
        "has_hallucination":          not det["deterministic_check_passed"] or llm_result.get("has_hallucination", False),
        "flagged_claims":             llm_result.get("flagged_claims", []),
        "confidence":                 llm_result.get("confidence", "high"),
        "deterministic_flags":        det["deterministic_flags"],
        "deterministic_check_passed": det["deterministic_check_passed"],
    }

    if combined["has_hallucination"]:
        print(f"  ⚠ Hallucination detected. Det flags: {det['deterministic_flags']}. LLM flags: {llm_result.get('flagged_claims')}")
    return {**state, "hallucination_report": combined}


# ── Node 6: Bias audit ────────────────────────────────────────────────────
def node_bias_audit(state: FinSightState) -> FinSightState:
    if budget_exceeded():
        return {**state, "error": _cost_state.get("budget_message", state.get("error", ""))}
    print("  [bias_audit] Checking for sentiment-driven bias...")
    system = "You are a bias auditor for AI-generated financial risk assessments."
    user = f"""
Risk report:       {json.dumps(state['risk_report'])}
News sentiment:    {json.dumps({k:v for k,v in state['news_sentiment_data'].get('data', {}).items() if k != 'classifications'})}
Fundamentals:      {json.dumps(state['fundamentals_data'].get('data', {}))}

Flag sentiment_bias_detected=true if the recommendation aligns with sentiment_score direction
but contradicts the fundamentals (e.g. 'hold' with negative sentiment but strong financials).

Return JSON: {{"sentiment_bias_detected": bool, "reasoning": "<one sentence>"}}
"""
    result = call_llm_json(system, user, BiasReport)
    return {**state, "bias_report": result}


# ── Node 7: Report refiner (used in loop) ─────────────────────────────────
def node_refine_report(state: FinSightState) -> FinSightState:
    if budget_exceeded():
        return {**state, "error": _cost_state.get("budget_message", state.get("error", ""))}
    iter_n = state.get("refinement_iter", 0) + 1
    print(f"  [refine_report] Refinement iteration {iter_n}...")
    system = "You are an expert financial analyst. Improve the risk report based on feedback."
    user = f"""
Original report:          {json.dumps(state['risk_report'])}
Judge feedback:           {json.dumps(state['judge_score'])}
Hallucination report:     {json.dumps(state.get('hallucination_report', {}))}
Bias report:              {json.dumps(state.get('bias_report', {}))}

Improve the report to address all flagged issues. Keep the same JSON schema.
Fix any flagged values. The composite_trend field must stay exactly as-is.

Return an improved RiskReport JSON with the same fields as the original.
"""
    result = call_llm_json(system, user, RiskReport)
    # Preserve deterministic fields through refinement
    if result and state["risk_report"].get("composite_trend"):
        result["composite_trend"] = state["risk_report"]["composite_trend"]
    return {**state, "risk_report": result, "refinement_iter": iter_n}


# ── Node 8: Compliance + final formatting ─────────────────────────────────
def node_finalize(state: FinSightState) -> FinSightState:
    print("  [finalize] Applying final deterministic gate, compliance filter, and generating final text...")
    rr  = state.get("risk_report", {})
    js  = state.get("judge_score", {})
    qd  = state.get("quant_data", {})
    hr  = state.get("hallucination_report", {})
    br  = state.get("bias_report", {})

    if rr:
        final_det = deterministic_hallucination_check(
            risk_report          = rr,
            market_data          = state.get("market_data", {}),
            iv_data              = state.get("iv_data", {}),
            fundamentals_data    = state.get("fundamentals_data", {}),
            news_sentiment_data  = state.get("news_sentiment_data", {}),
            macro_data           = state.get("macro_data", {}),
        )
        existing_flags = hr.get("deterministic_flags", [])
        hr = {
            **hr,
            "deterministic_check_passed": final_det["deterministic_check_passed"],
            "deterministic_flags": list(dict.fromkeys(existing_flags + final_det["deterministic_flags"])),
            "has_hallucination": hr.get("has_hallucination", False) or not final_det["deterministic_check_passed"],
        }
    if budget_exceeded():
        state = {**state, "error": _cost_state.get("budget_message", state.get("error", ""))}

    risk_emoji = {"low": "🟢", "moderate": "🟡", "high": "🔴"}.get(rr.get("risk_level", ""), "⚪")
    det_status = "✓ Passed" if hr.get("deterministic_check_passed", True) else f"⚠ Failed: {hr.get('deterministic_flags', [])}"

    report_text = f"""FinSight v2 Enhanced Risk Report
{'='*55}
Ticker:            {rr.get('ticker')}
Company:           {state['market_data'].get('data', {}).get('company_name', 'N/A')}
Risk Level:        {risk_emoji} {rr.get('risk_level', '').upper()}
Composite Score:   {rr.get('composite_score')} / 100
Composite Trend:   {rr.get('composite_trend', 'N/A')}
Recommendation:    {rr.get('recommendation', '').upper()}

Summary
{'-'*40}
{rr.get('summary')}

{'MD&A Grounding' if rr.get('mda_excerpt') else ''}
{('-'*40) if rr.get('mda_excerpt') else ''}
{rr.get('mda_excerpt', '') if rr.get('mda_excerpt') else ''}

Market Data
{'-'*40}
Current Price:     ${state['market_data'].get('data', {}).get('current_price', 'N/A')}
52-Week High:      ${state['market_data'].get('data', {}).get('52_week_high', 'N/A')}
52-Week Low:       ${state['market_data'].get('data', {}).get('52_week_low', 'N/A')}
Historical Vol:    {rr.get('volatility_pct')}%
Implied Vol:       {rr.get('implied_vol_pct', 'N/A')}%
P/E Ratio:         {rr.get('pe_ratio', 'N/A')}

Fundamentals
{'-'*40}
Debt/Equity:       {rr.get('debt_to_equity', 'N/A')}

Sentiment & Macro
{'-'*40}
Sentiment Score:   {rr.get('sentiment_score', 'N/A')} (-1 bearish → +1 bullish)
Sentiment Method:  {state['news_sentiment_data'].get('data', {}).get('method', 'N/A')}
Dominant Theme:    {rr.get('dominant_theme', 'N/A')}
Market Regime:     {rr.get('market_regime', 'N/A')}
VIX:               {state['macro_data'].get('data', {}).get('vix', 'N/A')}
10Y Yield:         {state['macro_data'].get('data', {}).get('ten_year_yield_pct', 'N/A')}%

Quant Signals
{'-'*40}
RSI Signal:        {qd.get('rsi_signal', 'N/A')}
Momentum:          {qd.get('momentum_signal', 'N/A')}
Macro-Adjusted:    {qd.get('macro_adjusted_signal', 'N/A')}
Technical Note:    {qd.get('technical_summary', 'N/A')}

Evaluation
{'-'*40}
Judge Score:       {js.get('score')}/5
Judge Reasoning:   {js.get('reasoning')}
Det. Pre-check:    {det_status}
Hallucination:     {'⚠ Detected' if hr.get('has_hallucination') else '✓ None detected'}
Bias Audit:        {'⚠ Bias detected' if br.get('sentiment_bias_detected') else '✓ No bias detected'}
Data Quality:      {rr.get('data_quality', 'N/A')}
Refinement Iters:  {state.get('refinement_iter', 0)}
"""
    report_text, redacted = apply_compliance_filter(report_text)
    if redacted:
        report_text += f"\n[Compliance: redacted {redacted}]"

    return {**state, "hallucination_report": hr, "final_report_text": report_text}


print("All node functions defined ✓")

## 9 · Build the LangGraph

In [ ]:
# ── Routing logic ─────────────────────────────────────────────────────────
def route_after_judge(state: FinSightState) -> str:
    if budget_exceeded():
        print("  [router] Budget exceeded → finalizing with partial results")
        return "finalize"
    score = state.get("judge_score", {}).get("score", 3)
    iters = state.get("refinement_iter", 0)

    if score >= JUDGE_THRESHOLD:
        return "hallucination_check"
    elif score == 1 or iters >= 2:
        print(f"  [router] Score={score}, iters={iters} → skipping to finalize")
        return "finalize"
    else:
        print(f"  [router] Score={score} below threshold → refining (iter {iters+1})")
        return "refine_report"


# ── Graph assembly ────────────────────────────────────────────────────────
builder = StateGraph(FinSightState)

builder.add_node("collect_data",        node_collect_data)
builder.add_node("quant_strategy",      node_quant_strategy)
builder.add_node("risk_synthesis",      node_risk_synthesis)
builder.add_node("judge",               node_judge)
builder.add_node("refine_report",       node_refine_report)
builder.add_node("hallucination_check", node_hallucination_check)
builder.add_node("bias_audit",          node_bias_audit)
builder.add_node("finalize",            node_finalize)

builder.set_entry_point("collect_data")
builder.add_edge("collect_data",   "quant_strategy")
builder.add_edge("quant_strategy", "risk_synthesis")
builder.add_edge("risk_synthesis", "judge")

builder.add_conditional_edges(
    "judge",
    route_after_judge,
    {
        "hallucination_check": "hallucination_check",
        "refine_report":       "refine_report",
        "finalize":            "finalize",
    },
)
builder.add_edge("refine_report",       "judge")
builder.add_edge("hallucination_check", "bias_audit")
builder.add_edge("bias_audit",          "finalize")
builder.add_edge("finalize",            END)

graph = builder.compile()
print("LangGraph compiled ✓")
print("Nodes:", list(builder.nodes.keys()))

## 10 · PDF Export

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle


def export_pdf(state: FinSightState, output_path: str = None) -> str:
    ticker = state["ticker"]
    rr     = state["risk_report"]
    if output_path is None:
        output_path = f"finsight_{ticker}_{datetime.utcnow().strftime('%Y%m%d_%H%M')}.pdf"

    doc    = SimpleDocTemplate(output_path, pagesize=A4,
                               leftMargin=2*cm, rightMargin=2*cm,
                               topMargin=2*cm, bottomMargin=2*cm)
    styles = getSampleStyleSheet()
    story  = []

    title_style = ParagraphStyle("Title", parent=styles["Title"],
                                 textColor=colors.HexColor("#6B21A8"), fontSize=20)
    story.append(Paragraph(f"FinSight v2 Enhanced — Risk Report: {ticker}", title_style))
    story.append(Spacer(1, 0.4*cm))

    risk_color = {"low": colors.green, "moderate": colors.orange, "high": colors.red}
    level = rr.get("risk_level", "unknown")
    story.append(Paragraph(
        f"<b>Risk Level:</b> <font color='{risk_color.get(level, colors.black).hexval()}'>"
        f"{level.upper()}</font>  |  "
        f"<b>Composite Score:</b> {rr.get('composite_score')}/100  |  "
        f"<b>Recommendation:</b> {rr.get('recommendation', '').upper()}",
        styles["Normal"]
    ))
    story.append(Spacer(1, 0.2*cm))
    trend_style = ParagraphStyle("Trend", parent=styles["Normal"],
                                 textColor=colors.HexColor("#7C3AED"), fontSize=9)
    story.append(Paragraph(f"<b>Composite Trend:</b> {rr.get('composite_trend', 'N/A')}", trend_style))
    story.append(Spacer(1, 0.3*cm))
    story.append(Paragraph(f"<i>{rr.get('summary')}</i>", styles["Normal"]))

    if rr.get("mda_excerpt"):
        story.append(Spacer(1, 0.3*cm))
        mda_style = ParagraphStyle("MDA", parent=styles["Normal"], fontSize=8,
                                   textColor=colors.HexColor("#374151"),
                                   borderPad=4, leftIndent=12)
        story.append(Paragraph(f"<b>MD&A:</b> {rr['mda_excerpt'][:400]}...", mda_style))

    story.append(Spacer(1, 0.5*cm))

    md  = state["market_data"].get("data", {})
    mac = state["macro_data"].get("data", {})
    qd  = state.get("quant_data", {})
    hr  = state.get("hallucination_report", {})
    table_data = [
        ["Field", "Value"],
        ["Current Price",   f"${md.get('current_price', 'N/A')}"],
        ["52W High / Low",  f"${md.get('52_week_high', 'N/A')} / ${md.get('52_week_low', 'N/A')}"],
        ["Historical Vol",  f"{rr.get('volatility_pct')}%"],
        ["Implied Vol",     f"{rr.get('implied_vol_pct', 'N/A')}%"],
        ["P/E Ratio",       str(rr.get('pe_ratio', 'N/A'))],
        ["Debt/Equity",     str(rr.get('debt_to_equity', 'N/A'))],
        ["Sentiment Score", str(rr.get('sentiment_score', 'N/A'))],
        ["Dominant Theme",  str(rr.get('dominant_theme', 'N/A'))],
        ["Market Regime",   str(rr.get('market_regime', 'N/A'))],
        ["VIX",             str(mac.get('vix', 'N/A'))],
        ["10Y Yield",       f"{mac.get('ten_year_yield_pct', 'N/A')}%"],
        ["RSI Signal",      str(qd.get('rsi_signal', 'N/A'))],
        ["Momentum",        str(qd.get('momentum_signal', 'N/A'))],
        ["Judge Score",     f"{state['judge_score'].get('score')}/5"],
        ["Det. Pre-check",  "✓ Passed" if hr.get("deterministic_check_passed", True) else "⚠ Failed"],
        ["Data Quality",    rr.get('data_quality', 'N/A')],
    ]
    t = Table(table_data, colWidths=[6*cm, 10*cm])
    t.setStyle(TableStyle([
        ("BACKGROUND",  (0,0), (-1,0), colors.HexColor("#6B21A8")),
        ("TEXTCOLOR",   (0,0), (-1,0), colors.white),
        ("FONTNAME",    (0,0), (-1,0), "Helvetica-Bold"),
        ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.HexColor("#F5F0FF"), colors.white]),
        ("GRID",        (0,0), (-1,-1), 0.5, colors.HexColor("#DDD6FE")),
        ("FONTSIZE",    (0,0), (-1,-1), 9),
        ("TOPPADDING",  (0,0), (-1,-1), 4),
        ("BOTTOMPADDING", (0,0), (-1,-1), 4),
    ]))
    story.append(t)
    story.append(Spacer(1, 0.5*cm))

    story.append(Paragraph("<b>Evaluation</b>", styles["Heading2"]))
    br = state.get("bias_report", {})
    story.append(Paragraph(
        f"Judge: {state['judge_score'].get('reasoning')} | "
        f"Det. Pre-check: {'✓ Passed' if hr.get('deterministic_check_passed', True) else '⚠ Failed'} | "
        f"Hallucination: {'⚠ Detected' if hr.get('has_hallucination') else '✓ None'} | "
        f"Bias: {'⚠ Detected' if br.get('sentiment_bias_detected') else '✓ None'}",
        styles["Normal"]
    ))
    story.append(Spacer(1, 0.5*cm))

    disc_style = ParagraphStyle("Disc", parent=styles["Normal"],
                                fontSize=7, textColor=colors.grey)
    story.append(Paragraph(
        "DISCLAIMER: This report is generated by FinSight AI for informational purposes only. "
        "It does not constitute financial advice or a recommendation to buy, hold, or sell any "
        "security. Always consult a qualified financial adviser.",
        disc_style
    ))

    doc.build(story)
    print(f"  PDF exported → {output_path}")
    return output_path


print("PDF export function defined ✓")

## 11 · Portfolio Mode

**ENHANCED:** Accepts a list of tickers, fans them all out through the existing graph in parallel,
then runs a correlation synthesis pass across all results.

In [ ]:
def run_portfolio(
    tickers: List[str],
    export_pdf_each: bool = False,
    max_workers: int = 4,
) -> dict:
    """
    Run the FinSight pipeline for a list of tickers in parallel, then
    synthesise a portfolio-level correlation/concentration report.

    Args:
        tickers:         List of ticker symbols, e.g. ["AAPL", "NVDA", "MSFT"]
        export_pdf_each: Whether to export a PDF for each individual ticker
        max_workers:     Parallel workers (keep low to respect rate limits)

    Returns:
        dict with keys: ticker_states, portfolio_synthesis, portfolio_summary_text
    """
    print(f"\n{'='*55}")
    print(f"  FinSight v2 Portfolio Mode — {tickers}")
    print(f"{'='*55}")

    base_state: FinSightState = {
        "ticker":               "",
        "market_data":          {},
        "iv_data":              {},
        "fundamentals_data":    {},
        "news_sentiment_data":  {},
        "macro_data":           {},
        "quant_data":           {},
        "risk_report":          {},
        "judge_score":          {},
        "hallucination_report": {},
        "bias_report":          {},
        "refinement_iter":      0,
        "final_report_text":    "",
        "error":                "",
    }

    ticker_states = {}

    def run_single(ticker: str):
        reset_cost_state()
        t0 = time.time()
        try:
            state = graph.invoke({**base_state, "ticker": ticker})
            elapsed = time.time() - t0
            print(f"  ✓ {ticker} done in {elapsed:.1f}s | composite={state['risk_report'].get('composite_score')} | {state['risk_report'].get('risk_level')}")
            if export_pdf_each:
                export_pdf(state)
            # Save to SQLite and ChromaDB
            save_report(ticker, state["risk_report"], state["judge_score"],
                        _cost_state["input_tokens"] + _cost_state["output_tokens"])
            chroma_store_report(state["risk_report"], state["judge_score"])
            return ticker, state
        except Exception as e:
            print(f"  ✗ {ticker} failed: {e}")
            return ticker, None

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(run_single, t): t for t in tickers}
        for f in concurrent.futures.as_completed(futures):
            ticker, state = f.result()
            ticker_states[ticker] = state

    # ── Portfolio synthesis pass ──────────────────────────────────────────
    # Build a compact summary of each ticker for the synthesis prompt
    ticker_summaries = []
    for t, state in ticker_states.items():
        if state is None:
            ticker_summaries.append(f"{t}: [pipeline error — no data]")
            continue
        rr = state["risk_report"]
        ticker_summaries.append(
            f"{t}: risk_level={rr.get('risk_level')}, composite={rr.get('composite_score')}, "
            f"recommendation={rr.get('recommendation')}, sector={state['market_data'].get('data', {}).get('sector', 'N/A')}, "
            f"market_regime={rr.get('market_regime')}, sentiment={rr.get('sentiment_score')}, "
            f"trend={rr.get('composite_trend', 'N/A')}"
        )

    print("\n  [portfolio] Running cross-ticker synthesis...")
    system = "You are a portfolio risk analyst. Synthesise risk signals across multiple holdings."
    user = f"""
Portfolio holdings analysis:
{chr(10).join(ticker_summaries)}

Identify:
1. concentration_warnings: any sector or regime concentration (e.g. "3 of 5 holdings are in semiconductors in a risk-off regime")
2. overall_portfolio_risk: aggregate risk level ('low', 'moderate', 'high')
3. correlation_notes: key observations about how these holdings may move together
4. portfolio_summary: two-sentence portfolio-level risk assessment and action item

Return JSON matching: {{concentration_warnings, overall_portfolio_risk, correlation_notes, portfolio_summary}}
"""
    synthesis = call_llm_json(system, user, PortfolioSynthesis)

    # Format portfolio summary text
    risk_emoji = {"low": "🟢", "moderate": "🟡", "high": "🔴"}.get(synthesis.get("overall_portfolio_risk", ""), "⚪")
    summary_lines = [
        f"\nFinSight v2 Portfolio Analysis",
        "=" * 55,
        f"Holdings:          {', '.join(tickers)}",
        f"Overall Risk:      {risk_emoji} {synthesis.get('overall_portfolio_risk', '').upper()}",
        "",
        "Concentration Warnings",
        "-" * 40,
    ]
    for w in synthesis.get("concentration_warnings", []):
        summary_lines.append(f"  • {w}")
    summary_lines += [
        "",
        "Correlation Notes",
        "-" * 40,
        synthesis.get("correlation_notes", "N/A"),
        "",
        "Portfolio Summary",
        "-" * 40,
        synthesis.get("portfolio_summary", "N/A"),
        "",
        "Individual Holdings",
        "-" * 40,
    ]
    for t, state in ticker_states.items():
        if state:
            rr = state["risk_report"]
            e  = {"low": "🟢", "moderate": "🟡", "high": "🔴"}.get(rr.get("risk_level", ""), "⚪")
            summary_lines.append(
                f"  {t}: {e} {rr.get('risk_level', '').upper()} | "
                f"score={rr.get('composite_score')} | "
                f"{rr.get('recommendation', '').upper()} | "
                f"trend={rr.get('composite_trend', 'N/A')}"
            )
        else:
            summary_lines.append(f"  {t}: ✗ error")

    portfolio_text = "\n".join(summary_lines)
    print(portfolio_text)

    return {
        "ticker_states":      ticker_states,
        "portfolio_synthesis": synthesis,
        "portfolio_summary_text": portfolio_text,
    }


print("Portfolio mode defined ✓")

## 12 · Run the Single-Ticker Pipeline

In [ ]:
# ── Configure your run here ───────────────────────────────────────────────
TICKER     = "AAPL"     # Change to any valid ticker, e.g. "TSLA", "NVDA", "BRK-B"
EXPORT_PDF = True       # Set False to skip PDF generation

reset_cost_state()

initial_state: FinSightState = {
    "ticker":               TICKER,
    "market_data":          {},
    "iv_data":              {},
    "fundamentals_data":    {},
    "news_sentiment_data":  {},
    "macro_data":           {},
    "quant_data":           {},
    "risk_report":          {},
    "judge_score":          {},
    "hallucination_report": {},
    "bias_report":          {},
    "refinement_iter":      0,
    "final_report_text":    "",
    "error":                "",
}

print(f"\n{'='*55}")
print(f"  FinSight v2 Enhanced Pipeline — {TICKER}")
print(f"{'='*55}")
t_start = time.time()

final_state = graph.invoke(initial_state)

elapsed = time.time() - t_start
print(f"\n{'='*55}")
print(f"  Pipeline complete in {elapsed:.1f}s")
print(f"  Cost: {cost_summary()}")
print(f"{'='*55}")

In [ ]:
print(final_state["final_report_text"])

In [ ]:
# ── Save to SQLite, ChromaDB, and optionally export PDF ───────────────────
row_id = save_report(
    ticker      = TICKER,
    risk_report = final_state["risk_report"],
    judge_score = final_state["judge_score"],
    token_cost  = _cost_state["input_tokens"] + _cost_state["output_tokens"],
)
print(f"Report saved to SQLite (row_id={row_id})")

chroma_store_report(final_state["risk_report"], final_state["judge_score"])

if EXPORT_PDF:
    pdf_path = export_pdf(final_state)
    print(f"PDF ready: {pdf_path}")

## 13 · Run Portfolio Mode

In [ ]:
# ── Configure your portfolio here ────────────────────────────────────────
PORTFOLIO_TICKERS = ["AAPL", "NVDA", "MSFT"]   # Add up to ~6 tickers

portfolio_result = run_portfolio(
    tickers         = PORTFOLIO_TICKERS,
    export_pdf_each = False,    # Set True to get a PDF per ticker
    max_workers     = 3,        # Lower if hitting Groq rate limits
)

## 14 · Report History, Trend View & ChromaDB Search

In [ ]:
# ── View SQLite report history ────────────────────────────────────────────
history = get_report_history(TICKER, limit=5)
print(f"Last {len(history)} runs for {TICKER}:")
for h in history:
    rr = h["risk_report"]
    js = h["judge_score"]
    print(
        f"  {h['run_at']}  |  {rr.get('ticker')}  "
        f"{rr.get('risk_level', '').upper()}  score={rr.get('composite_score')}  "
        f"judge={js.get('score')}/5  |  trend={rr.get('composite_trend', 'N/A')}"
    )

In [ ]:
# ── ChromaDB semantic search ──────────────────────────────────────────────
# Find reports matching a risk theme across all tickers
print("Semantic search — 'high risk regulatory':")
results = chroma_search("high risk regulatory", n_results=3)
for r in results:
    m = r["metadata"]
    print(f"  {m.get('ticker')} | {m.get('risk_level')} | score={m.get('composite_score')} | {m.get('run_at')[:10]}")
    print(f"    dist={r['distance']:.3f} | {r['document'][:120]}...")
    print()

In [ ]:
# ── Filtered ChromaDB search ─────────────────────────────────────────────
# Filter to only risk-off regime reports
print("Filtered search — only risk-off regime, query 'avoid semiconductor':")
results = chroma_search(
    query="avoid semiconductor",
    n_results=3,
    filter_metadata={"market_regime": "risk-off"}
)
for r in results:
    m = r["metadata"]
    print(f"  {m.get('ticker')} | {m.get('recommendation')} | regime={m.get('market_regime')}")

## 15 · Eval Harness

In [ ]:
EVAL_CASES = [
    {"id": "eval_001", "ticker": "AAPL",       "note": "Baseline happy path"},
    {"id": "eval_002", "ticker": "TSLA",       "note": "High-volatility; expect risk_level=high"},
    {"id": "eval_003", "ticker": "BRK-B",      "note": "Low-vol, high-fundamental quality"},
    {"id": "eval_004", "ticker": "INVALIDXYZ", "note": "Invalid ticker; graceful error"},
    {"id": "eval_005", "ticker": "GME",        "note": "Meme stock; sentiment-vs-fundamental bias"},
]

eval_results = []
base_state_eval: FinSightState = {
    "ticker": "", "market_data": {}, "iv_data": {}, "fundamentals_data": {},
    "news_sentiment_data": {}, "macro_data": {}, "quant_data": {}, "risk_report": {},
    "judge_score": {}, "hallucination_report": {}, "bias_report": {},
    "refinement_iter": 0, "final_report_text": "", "error": "",
}

for case in EVAL_CASES:
    print(f"\n[EVAL {case['id']}] {case['ticker']} — {case['note']}")
    reset_cost_state()
    t0 = time.time()
    try:
        state = graph.invoke({**base_state_eval, "ticker": case["ticker"]})
        elapsed = time.time() - t0
        rr = state["risk_report"]
        hr = state.get("hallucination_report", {})
        result = {
            "id":                    case["id"],
            "ticker":                case["ticker"],
            "risk_level":            rr.get("risk_level"),
            "composite":             rr.get("composite_score"),
            "composite_trend":       rr.get("composite_trend"),
            "judge_score":           state["judge_score"].get("score"),
            "hallucination":         hr.get("has_hallucination"),
            "det_check_passed":      hr.get("deterministic_check_passed", True),
            "data_quality":          rr.get("data_quality"),
            "latency_s":             round(elapsed, 1),
            "cost_usd":              round(_cost_state["total_usd"], 4),
            "pass": (
                state["judge_score"].get("score", 0) >= 3 and
                not hr.get("has_hallucination", False) and
                hr.get("deterministic_check_passed", True) and
                elapsed < 60
            ),
        }
        print(f"  ✓ {result}")
    except Exception as e:
        result = {"id": case["id"], "ticker": case["ticker"], "pass": False, "error": str(e)}
        print(f"  ✗ Error: {e}")
    eval_results.append(result)

print("\n" + "="*55)
print("EVAL SUMMARY")
print("="*55)
passed = sum(1 for r in eval_results if r.get("pass"))
print(f"{passed}/{len(eval_results)} cases passed")
for r in eval_results:
    status = "✓" if r.get("pass") else "✗"
    det    = "✓ det" if r.get("det_check_passed", True) else "⚠ det"
    print(f"  {status} {r['id']} ({r['ticker']}) {det}")